In [2]:
import torch
import random

SEED = 69
torch.manual_seed(seed=SEED)
random.seed(SEED)

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu'

print(f'Using {device} device')


Using cuda device


### _Convert Unicode to ASCII_

In [3]:
import string
import unicodedata

allowed_characters = string.ascii_letters + " .,;'_"
n_letters = len(allowed_characters)

def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
        and c in allowed_characters
    )  

In [4]:
print (f"converting 'Ślusàrski' to {unicodeToAscii('Ślusàrski')}")

converting 'Ślusàrski' to Slusarski


### Convert to Tensors

Now that we have all the names organized, we need to turn them into Tensors to make any use of them.

To represent a single letter, we use a “one-hot vector” of size `<1 x n_letters>`. A one-hot vector is filled with 0s except for a 1 at index of the current letter, e.g. `"b" = <0 1 0 0 0 ...>`.

To make a word we join a bunch of those into a 2D matrix `<line_length x 1 x n_letters>`.

That extra 1 dimension is because PyTorch assumes everything is in batches - we’re just using a batch size of 1 here.

In [5]:
# find letter from all_letters, e.g. 'a' = 0
def letterToIndex(letter):
    # return our out-of-vocabulary character if we encounter a letter unknown to our model
    if letter not in allowed_characters:
        return allowed_characters.find('_')
    else:
        return allowed_characters.find(letter)

# Turn a line into a 
# or an array of one-hot letter vectors
def lineToTensor(line):
    tensor = torch.zeros(len(line), 1, n_letters)
    for li, letter in enumerate(line):
        tensor[li][0][letterToIndex(letter)] = 1
    return tensor


notice that the first position in the tensor = 1

In [6]:
print(f"The index of letter 'a' is {letterToIndex('a')}")
print (f"The letter 'a' becomes {lineToTensor('a')}")

The index of letter 'a' is 0
The letter 'a' becomes tensor([[[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0.]]])


notice 'A' sets the 27th index to 1

In [7]:
print(f"The index of letter 'A' is {letterToIndex('A')}")
print(f"The index of letter 'h' is {letterToIndex('h')}")
print(f"The index of letter 'n' is {letterToIndex('n')}")
print (f"The name 'Ahn' becomes {lineToTensor('Ahn')}")

The index of letter 'A' is 26
The index of letter 'h' is 7
The index of letter 'n' is 13
The name 'Ahn' becomes tensor([[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0.]]])


This is normal text, but <span style="color:red">this text is red</span> and <span style="color:#007bff">this text is blue</span>.


Next, we need to combine all our examples into a dataset so we can train, test and validate our models. For this, we will use the Dataset and DataLoader classes to hold our dataset. Each Dataset needs to implement three functions: <span style='color:#87CEEB;'>`__init__`, `__len__`, `__getitem__` </span>.

In [8]:
from io import open
import glob, os, time, torch
from torch.utils.data import Dataset

class NamesDataset(Dataset):
    def __init__(self, data_dir):
        super().__init__()
        self.data_dir = data_dir # for prevenance of the dataset
        self.load_time = time.localtime # for provenance of the dataset
        
        labels_set = set() # set of all classes

        self.data = list()
        self.data_tensors = list()
        self.labels = list()
        self.labels_tensors = list()

        # real all the ``.txt`` files i nthe specified directory
        text_files = glob.glob(os.path.join(self.data_dir, '*.txt'))

        for filename in text_files:
            label = os.path.splitext(os.path.basename(filename))[0]
            labels_set.add(label)
            lines = open(filename, encoding='utf-8').read().strip().split('\n')

            for name in lines:
                self.data.append(name)
                self.data_tensors.append(lineToTensor(name))
                self.labels.append(label)

        # cache the tensor representation of the labels
        self.labels_uniq = list(labels_set)
        for idx in range(len(self.labels)):
            temp_tensor = torch.tensor([self.labels_uniq.index(self.labels[idx])], dtype=torch.long)
            self.labels_tensors.append(temp_tensor)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        data_item = self.data[index]
        data_label = self.labels[index]
        data_tensor = self.data_tensors[index]
        label_tensor = self.labels_tensors[index]
        
        return label_tensor, data_tensor, data_label, data_item


Here we can load our example data into the NamesDataset

In [9]:
all_data = NamesDataset(os.path.join(os.path.expandvars('$HOME'),'Dev/pytorch-tutorial/data/names'))

print(f'loaded {len(all_data)} items of data')
print(f'example = {all_data[0]}')

loaded 20074 items of data
example = (tensor([5]), tensor([[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0.]],

        [[0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0.]]]), 'Czech', 'Abl')


Split train set and test set

In [10]:
train_set, test_set = torch.utils.data.random_split(all_data, [0.85, 0.15], generator=torch.Generator(device='cpu'))
print(f'training examples = {len(train_set)}, validation examples = {len(test_set)}')

training examples = 17063, validation examples = 3011


### _Creating the Network_

Before autograd, creating a recurrent neural network in Torch involved cloning the parameters of a layer over several timesteps. The layers held hidden state and gradients which are now entirely handled by the graph itself. This means you can implement a RNN in a very “pure” way, as regular feed-forward layers.

This CharRNN class implements an RNN with three components. First, we use the nn.RNN implementation. Next, we define a layer that maps the RNN hidden layers to our output. And finally, we apply a softmax function. Using nn.RNN leads to a significant improvement in performance, such as cuDNN-accelerated kernels, versus implementing each layer as a nn.Linear. It also simplifies the implementation in forward().

In [11]:
from torch import nn
from torch.nn import functional as F 

class CharacterRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.rnn = nn.RNN(input_size=input_size, hidden_size=hidden_size)
        self.h2o = nn.Linear(in_features=hidden_size, out_features=output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, line_tensor):
        rnn_out, hidden = self.rnn(line_tensor)
        output = self.h2o(hidden[0])
        output = self.softmax(output)
        return output

We create a RNN

In [12]:
n_hidden = 128
rnn = CharacterRNN(input_size=n_letters, hidden_size=n_hidden, output_size=len(all_data.labels_uniq))
print(rnn)

CharacterRNN(
  (rnn): RNN(58, 128)
  (h2o): Linear(in_features=128, out_features=18, bias=True)
  (softmax): LogSoftmax(dim=1)
)


After creating a model, we can pass our Tensor to the RNN to obtain a predicted output. Subsequently, we use a helper function, `label_from_output`, to derive a text label for the class

In [13]:
def label_from_output(output, output_labels):
    _, top_i = output.topk(1)
    label_i = top_i[0].item()
    return output_labels[label_i], label_i


input = lineToTensor('Albert')
output = rnn(input) # this is equivalent to output = rnn.forward(input)
print(output)
print(label_from_output(output, all_data.labels_uniq))


tensor([[-2.9934, -2.7898, -2.7554, -2.9619, -2.6230, -2.8930, -2.8716, -2.8466,
         -2.9618, -2.8131, -2.9513, -2.9728, -3.0150, -2.8688, -2.8905, -2.9214,
         -3.0899, -2.9077]], grad_fn=<LogSoftmaxBackward0>)
('Dutch', 4)


# **_Training_**

#### **_Training the Network_**

Now all it takes to train this network is show it a bunch of examples, have it make guesses, and tell it if it’s wrong.

We do this by defining a `train()` function which trains the model on a given dataset using minibatches. RNNs RNNs are trained similarly to other networks; therefore, for completeness, we include a batched training method here. The loop (`for i in batch`) computes the losses for each of the items in the batch before adjusting the weights. This operation is repeated until the number of epochs is reached.

In [14]:
import numpy as np


def train(
    rnn: CharacterRNN,
    training_data,
    n_epoch: int = 10,
    n_batch_size: int = 64,
    report_every: int = 50,
    learning_rate: float = 0.2,
    criterion=nn.NLLLoss(),
):
    """
    Learn on a batch of training_data for a specified number of iterations and reporting thresholds
    """

    # keep track of losses for plotting
    current_loss = 0
    all_losses = list()
    rnn.train()
    optimizer = torch.optim.SGD(rnn.parameters(), lr=learning_rate)

    print(f'training on dataset with n = {len(training_data)}')

    for iter in range(0, n_epoch):
        rnn.zero_grad()  # clear the gradients

        # create some minibatches
        # we cannot use dataloaders because of our names is a different length
        batches = list(range(len(training_data)))
        random.shuffle(batches)
        batches = np.array_split(batches, len(batches) // n_batch_size)

        for _, batch in enumerate(batches):
            batch_loss = 0

            for i in batch:
                (label_tensor, text_tensor, label, text) = training_data[i]
                output = rnn.forward(text_tensor)
                loss = criterion(output, label_tensor)
                batch_loss += loss

            # optimize parameters
            batch_loss.backward()
            torch.nn.utils.clip_grad_norm_(rnn.parameters(), 3)
            optimizer.step()
            optimizer.zero_grad()

            current_loss += batch_loss.item() / len(batch)

        all_losses.append(current_loss/len(batches))

        if iter % report_every == 0:
            print(f'')
        current_loss = 0
    return all_losses

We can now train a dataset with minibatches for a specified number of epochs. The number of epochs for this example is reduced to speed up the build. You can get better results with different parameters.

In [15]:
start = time.time()
all_losses = train(rnn, train_set, n_epoch=27, learning_rate=0.15, report_every=5)
end = time.time()
print(f"training took {end-start}s")

training on dataset with n = 17063






training took 135.00988960266113s
